# 01 · Profile designer — every knob under its family

The 2-D profile of BladeGen lives in the **unstaggered (m′, θ) plane**: LE point at (0, 0), TE point at
(1, 0), suction side (SS) above. It is built by the C++ engine exactly as the blade2d Python engine did
(DLR-style characteristic points, cubic B-spline SS, thickness-coupled PS), with **C2 / G2 connectors** at
the leading and trailing edges.

**What you see on the dashboard**

| panel | content |
|---|---|
| top | the profile: SS blue, PS orange, LE connector green, TE connector red; control polygons dashed; characteristic points black |
| zooms | LE and TE close up (where the connectors meet the sides) |
| thickness | measured normal thickness (SS normal → PS) and the thickness law t(ξ) |
| curvature | **curvature along SS and PS, from the LE point to the TE point** (the plot the Python engine was missing) |
| badge | the geometry check: **✗ red = ERROR**, ⚠ orange = warning — the geometry is still built and drawn |

Errors are marked where they occur (red ✗ on the profile and on the curvature plot).

In [ ]:
%matplotlib inline
import os, sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# the bladegen package lives in <build>/python (cmake --preset python); BLADEGEN_PY overrides
REPO = Path.cwd().resolve().parents[1]
for cand in [os.environ.get("BLADEGEN_PY", ""), REPO / "build" / "python" / "python"]:
    if cand and Path(cand).is_dir() and str(cand) not in sys.path:
        sys.path.insert(0, str(cand))
import bladegen as bg
import bladegen.notebook as nb
OUT = Path(os.environ.get("BLADEGEN_TEST_OUTPUT", REPO / "build" / "notebook_output"))
OUT.mkdir(parents=True, exist_ok=True)
print("BladeGen", bg.__version__, "| widgets:", nb.HAVE_WIDGETS)

## Interactive designer

The knobs are grouped by family — **Leading edge, Trailing edge, Suction side, Thickness distribution,
Pressure side, Connector** — each knob sits under its family heading. Every change rebuilds the profile in
C++ (≈ 1 ms) and redraws the dashboard. Try `LE connector tension` → 6 to see a loop flagged as an
error, or pull `SS CP2 theta` down until the pressure side crosses the suction side.

In [ ]:
knobs = nb.interactive_profile(bg.ProfileParams.naca65_1210())

## What the curvature check found in the Python engine's NACA 65 fit

`naca65_1210_blade2d()` is the **blade2d fit** (its `naca65_1210_blade_params.json`) with the blade2d
connector (`G2Legacy`): the geometry the Python engine produced. Its TE radius sits on the lower bound
(0.000255 chord) because NACA 65 has a *cusped* trailing edge; the TE connector then has to form an
**S-bend** a few micrometres long — invisible in a profile plot, obvious in the curvature plot, and flagged
as an error. `naca65_1210()` is BladeGen's own refit with a fairness penalty: same accuracy, no defect.

In [ ]:
for p, t in [(bg.ProfileParams.naca65_1210_blade2d(), "blade2d fit + blade2d connector (Python geometry)"),
             (bg.ProfileParams.naca65_1210(), "BladeGen fairness-weighted refit (default)")]:
    fig, prof, q = nb.profile_dashboard(p, title=t)
    plt.show()
    print(q["summary"])

## The three connector modes

* **Fair** (default): G2 at both junctions (position, unit tangent and curvature equal to the side's) and
  the *fairest* such curve — a degree-5 B-spline with 4 interior knots through the edge point that
  minimises the jerk energy ∫|C‴|² (one KKT solve).
* **C2Parametric**: copies the sides' first *and second* derivatives, so the closed loop is one C2 curve
  in a single parameter — exact, but the copied tangential acceleration can make a small edge spike.
* **G2Legacy**: blade2d's two-span quintic (the Python engine).

Below: the LE curvature of the same profile with each mode.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
base = bg.ProfileParams.naca65_1210()
for mode, ls in [("Fair", "-"), ("C2Parametric", "--"), ("G2Legacy", ":")]:
    p = base.copy(); p.connector = getattr(bg.ConnectorMode, mode)
    if mode != "Fair":
        p.le.tension, p.te.tension = 0.28, 0.20
    q = bg.check_quality(bg.Profile2D(p))
    for k, side, c in ((0, "ss", nb.C_SS), (1, "ps", nb.C_PS)):
        s = q[side]; x = s["s"] / s["s"][-1]
        ax[k].plot(x, s["kappa"], ls, color=c, label=f"{mode}: {q['errors']} err / {q['warnings']} warn")
for a, t in zip(ax, ["suction side", "pressure side"]):
    a.set_xlim(0, 0.05); a.set_yscale("symlog", linthresh=1); a.grid(alpha=.3); a.legend(fontsize=8)
    a.set_title(f"{t}: curvature near the LE"); a.set_xlabel("arc length / side length")
plt.show()

## Saving and loading

Profiles are flat JSON files. `read_profile_json` also reads blade2d's `<name>_blade_params.json`
directly (its keys are recognised), so every profile fitted with the Python tools can be loaded.

In [ ]:
p = knobs.params
bg.write_profile_json(p, str(OUT / "my_profile.json"))
q = bg.read_profile_json(str(OUT / "my_profile.json"))
old = bg.read_profile_json(str(REPO / "tests" / "data" / "naca65_1210_blade_params.json"))  # a blade2d file
print("round trip ok:", all(abs(p.get(k) - q.get(k)) < 1e-12 for k in p.to_dict() if isinstance(p.to_dict()[k], float)))
print("blade2d file -> connector", old.connector, "| LE radius", old.le.radius)